In [1]:
import pandas as pd
from pathlib import Path

processed_path = Path("../../data/processed")

final_df = pd.read_parquet(processed_path / "final_dataset.parquet")

In [2]:
"SUBJECT_ID" in final_df.columns

False

In [3]:
cohort_df = pd.read_parquet("../../data/processed/adult_icu_cohort_first24h.parquet")

cohort_df.columns

Index(['SUBJECT_ID', 'HADM_ID', 'ICUSTAY_ID', 'INTIME'], dtype='str')

In [4]:
patient_map = cohort_df[["ICUSTAY_ID", "SUBJECT_ID"]]

In [5]:
patient_map.head()

,ICUSTAY_ID,SUBJECT_ID
0,280836,268
1,206613,269
2,220345,270
3,249196,271
4,210407,272


In [6]:
final_df = final_df.merge(
    patient_map,
    on="ICUSTAY_ID",
    how="left"
)

In [7]:
final_df[["ICUSTAY_ID", "SUBJECT_ID"]].head()

,ICUSTAY_ID,SUBJECT_ID
0,280836,268
1,206613,269
2,220345,270
3,249196,271
4,210407,272


In [8]:
final_df["SUBJECT_ID"].isna().sum()

np.int64(0)

In [9]:
y = final_df["HOSPITAL_EXPIRE_FLAG"]

subject_id = final_df["SUBJECT_ID"]

In [10]:
X = final_df.drop(columns=[
    "HOSPITAL_EXPIRE_FLAG",
    "SUBJECT_ID",
    "ICUSTAY_ID"
])

In [11]:
from sklearn.model_selection import GroupShuffleSplit

In [12]:
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

In [13]:
train_idx, test_idx = next(
    splitter.split(X, y, groups=subject_id)
)

In [14]:
X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

## Train-Test Split

The final dataset was prepared for machine learning by separating predictors, target, and patient identifiers.

- `HOSPITAL_EXPIRE_FLAG` was defined as the target variable.
- `SUBJECT_ID` was used only to keep multiple ICU stays from the same patient in the same split.
- `ICUSTAY_ID` and `SUBJECT_ID` were excluded from model predictors.
- The data was split into approximately 80% training and 20% test sets.
- `GroupShuffleSplit` was used to prevent ICU stays from the same patient from appearing in both training and test sets.

In [15]:
categorical_cols = [
    "gender",
    "admission_type",
    "admission_location"
]


numeric_cols = X_train.columns.difference(categorical_cols).tolist()

In [16]:
from sklearn.impute import SimpleImputer
numeric_imputer = SimpleImputer(strategy="median")

from sklearn.preprocessing import OneHotEncoder
categorical_encoder = OneHotEncoder(handle_unknown="ignore")

from sklearn.compose import ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_imputer, numeric_cols),
        ("categorical", categorical_encoder, categorical_cols)
    ]
)

In [17]:
from sklearn.model_selection import StratifiedGroupKFold

subject_id_train = subject_id.iloc[train_idx]

cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [18]:
print("Categorical columns:")
print(categorical_cols)

print("\nNumeric column count:")
print(len(numeric_cols))

print("\nCategorical dtypes:")
print(X_train[categorical_cols].dtypes)

print("\nTrain shape:")
print(X_train.shape)

Categorical columns:
['gender', 'admission_type', 'admission_location']

Numeric column count:
80

Categorical dtypes:
gender                str
admission_type        str
admission_location    str
dtype: object

Train shape:
(36169, 83)


## Preprocessing and Cross-Validation Setup

- The training data was separated into numerical and categorical feature groups.
- Numerical features were assigned a median imputation strategy using `SimpleImputer`.
- Categorical features were assigned a `OneHotEncoder` with unknown-category handling.
- Both preprocessing steps were combined using `ColumnTransformer`.
- `StratifiedGroupKFold` was prepared for 5-fold cross-validation.
- Patient-level grouping will use `SUBJECT_ID` so that multiple ICU stays from the same patient remain in the same fold.
- A final sanity check confirmed 3 categorical features, 80 numerical features, and a training shape of `(36169, 83)`.

In [19]:
subject_id_test = subject_id.iloc[test_idx]

train_data = X_train.copy()
train_data["SUBJECT_ID"] = subject_id_train
train_data["HOSPITAL_EXPIRE_FLAG"] = y_train

test_data = X_test.copy()
test_data["SUBJECT_ID"] = subject_id_test
test_data["HOSPITAL_EXPIRE_FLAG"] = y_test

train_data.to_parquet("../../data/processed/ml_train.parquet")
test_data.to_parquet("../../data/processed/ml_test.parquet")